# Módulo de Control de Calidad, Tipificación y Balance de Muestras (`heavystats`)

Este cuaderno interactivo implementa y documenta la metodología formal de **control de calidad de datos**, **catalogación de variables**, **preprocesamiento y estandarización**, y **evaluación diagnóstica de sesgo de selección** (balance entre población $N=48$ y muestra analítica $n=20$) de acuerdo con las secciones 1 y 2 del protocolo de investigación de metales pesados en población infantil (normas STROBE / STREGA / ICMJE).

### Objetivos Metodológicos y Estadísticos
1. **Carga y Verificación Estructural de Datos**: Carga robusta de la base de datos epidemiológica bruta y verificación de codificación UTF-8/Latin-1, separadores decimales y tipos nativos.
2. **Auditoría Sistemática de Calidad (10 Reglas)**: Ejecución de la rutina automatizada `validate_data()` para comprobar dimensiones, completitud en la muestra analítica, ausencia de duplicados, validez de tipos, límites biológicos/físicos (edad 6-10 años, peso 5-150 kg, talla 50-220 cm), límites de detección analíticos ($\text{LOD}$) y variabilidad mínima de predictores.
3. **Tipificación y Catalogación de Variables**: Inspección de la arquitectura del dataset (`columns_type()`) y construcción de la tabla catalogada de variables (`variables_table()`) clasificando covariables en continuas, discretas, categóricas y de respuesta múltiple.
4. **Pipeline de Preprocesamiento y Normalización**:
   - Estandarización de variantes tipográficas binarias (`standardize_boolean_columns()`).
   - Desagregación de respuestas múltiples delimitadas por punto y coma (`;`) en indicadores binarios independientes (`desaggregate_multiple_responses()`).
   - Codificación ordinal de frecuencias alimentarias (`encode_dietary_frequencies()`).
5. **Partición de Población y Muestra Analítica**: Separación formal entre la cohorte total ($N=48$) y la muestra con mediciones de metales en sangre ($n=20$, identificados por `Muestra_Codificada`) mediante `get_analytical_sample()`, y filtrado de metales aislados (`select_metal()`).
6. **Diagnóstico de Sesgo de Selección y Balance de Muestras**: Evaluación de desequilibrios basales mediante la **Diferencia Media Estandarizada (SMD)** ($|\text{SMD}| > 0.10$ leve, $|\text{SMD}| > 0.25$ moderado/severo), prueba $t$ de Welch para variables continuas y Prueba Exacta de Fisher / Chi-cuadrado para categóricas con `compare_groups()`.


--- 
## 1. Carga y Verificación Estructural de Datos Brutos

Cargamos el dataset epidemiológico bruto mediante `load_data()`, el cual resuelve automáticamente discrepancias de codificación de caracteres especiales (acentos, eñes) y estandariza los tipos de datos iniciales.


In [1]:
import sys
from pathlib import Path

# Vincular la carpeta 'src' del paquete heavystats
sys.path.insert(0, str(Path.cwd().resolve().parents[0]))

import pandas as pd
import numpy as np
import heavystats as hs

# 1. Cargar datos brutos
df_raw = hs.load_data()
print(f"Versión de HeavyStats: {hs.__version__}")
print(f"Dataset bruto cargado: {df_raw.shape[0]} observaciones y {df_raw.shape[1]} columnas.")

<frozen importlib._bootstrap>:491: Warning: Numpy built with MINGW-W64 on Windows 64 bits is experimental, and only available for 
testing. You are advised not to use it for production. 

CRASHES ARE TO BE EXPECTED - PLEASE REPORT THEM TO NUMPY DEVELOPERS


Versión de HeavyStats: 0.2.0.dev1
Dataset bruto cargado: 48 observaciones y 39 columnas.


--- 
## 2. Control de Calidad y Validación de Reglas Lógicas (10 Reglas)

La función `validate_data()` ejecuta una batería de **10 pruebas automáticas** para certificar que los datos cumplen los estándares de consistencia lógica, matemática y física requeridos antes de cualquier inferencia estadística:

| N° | Criterio de Control | Regla de Validación |
| :---: | :--- | :--- |
| 1 | **Dimensiones** | Exactitud en el número de filas (48) y columnas requeridas. |
| 2 | **Población y Muestra** | Presencia de $N=48$ encuestados y $n=20$ pacientes con analíticas de sangre. |
| 3 | **Columnas Críticas** | Existencia de identificadores, metales, riesgo y covariables basales. |
| 4 | **Valores Duplicados** | Comprobación de unicidad en registros de pacientes. |
| 5 | **Completitud** | Ausencia estricta de valores nulos ($0\%$) en la muestra analítica ($n=20$). |
| 6 | **Tipos de Datos** | Formato numérico en concentraciones y antropometría. |
| 7 | **Valores Físicos e Imposibles** | Edad $\in [6, 10]$, Peso $\in [5, 150]$, Altura $\in [50, 220]$, Concentraciones $\ge 0$. |
| 8 | **Codificación Categórica** | Homogeneidad en categorías cualitativas. |
| 9 | **Límites de Detección (LOD)** | Valores por encima de los límites de sensibilidad instrumental. |
| 10 | **Variabilidad Mínima** | Existencia de al menos 2 niveles en cada variable predictora de riesgo. |


In [2]:
# Ejecutar validación integral de calidad de datos
report_qc = hs.validate_data(df_raw)
report_qc

Estado,Criterio,Detalle / Mensaje
✓ Correcto,Dimensiones,48 filas y 39 columnas.
✓ Correcto,Población y Muestra,Población N=48 | Muestra n=20.
✓ Correcto,Columnas Críticas,Todas las columnas requeridas están presentes.
✓ Correcto,Valores Duplicados,Sin registros duplicados en el dataset.
✓ Correcto,Completitud,Sin valores nulos en los datos de la muestra.
✓ Correcto,Tipos de Datos,Las columnas críticas tienen un formato válido.
✓ Correcto,Valores Imposibles,Todos los límites lógicos y físicos se cumplen.
✓ Correcto,Codificación Categorías,Codificación de variables categóricas consistente.
✓ Correcto,Límites de Detección,Las concentraciones superan los límites de detección.
✓ Correcto,Variabilidad Mínima,Todas las variables de clasificación tienen variabilidad.


--- 
## 3. Tipificación y Catalogación de Variables

Para planificar adecuadamente los análisis descriptivos e inferenciales, exploramos la distribución de variables por tipo de dato nativo (`columns_type()`) y generamos la tabla formal de clasificación metodológica (`variables_table()`).


In [3]:
# Reporte de tipos de datos y nulos detectados en el dataset bruto
report_types = hs.columns_type(df_raw)
report_types

Tipo de Dato,N° Columnas,Variables Asignadas
Int64,1,Muestra_Codificada ⚠️ 28 nulos
float64,5,Peso_kg ⚠️ 28 nulosAltura_cm ⚠️ 28 nulosPlomo_ug_dL ⚠️ 28 nulosMercurio_ug_L ⚠️ 28 nulosCadmio_ug_L ⚠️ 28 nulos
int64,2,EdadScore_Riesgo
str,31,SexoSectorInstitucionEs_ExpuestoRiesgo_PbRiesgo_HgRiesgo_CdExposicion_TalleresExposicion_IndustriasExposicion_LugaresAlim_CerealesAlim_LeguminosasAlim_TuberculosAlim_CarnesAlim_PescadosAlim_BebidasAlim_HuevosAlim_LacteosAlim_FrutasAlim_VegetalesAlim_AzucarAlim_GrasasAlim_ChocolateSalud_FumaSalud_ActividadSalud_BombillosSalud_TechoSalud_JoyeriaSalud_TransporteSalud_AguaSalud_Suplementos


In [4]:
# Tabla estructurada clasificando cada variable como Numérica, Categórica u Otro
report_vars = hs.variables_table(df_raw)
report_vars

Clasificación,N° Columnas,Variables Asignadas
Categórica,31,SexoSectorInstitucionEs_ExpuestoRiesgo_PbRiesgo_HgRiesgo_CdExposicion_TalleresExposicion_IndustriasExposicion_LugaresAlim_CerealesAlim_LeguminosasAlim_TuberculosAlim_CarnesAlim_PescadosAlim_BebidasAlim_HuevosAlim_LacteosAlim_FrutasAlim_VegetalesAlim_AzucarAlim_GrasasAlim_ChocolateSalud_FumaSalud_ActividadSalud_BombillosSalud_TechoSalud_JoyeriaSalud_TransporteSalud_AguaSalud_Suplementos
Numérica,8,Muestra_CodificadaEdadPeso_kgAltura_cmScore_RiesgoPlomo_ug_dLMercurio_ug_LCadmio_ug_L


--- 
## 4. Pipeline de Limpieza, Estandarización y Desagregación

Para habilitar el modelado cuantitativo, aplicamos las transformaciones estandarizadas del paquete:
1. **`standardize_boolean_columns()`**: Estandariza respuestas afirmativas/negativas con variantes tipográficas (`sí`, `si`, `sI`, `no`) a categorías estándar en mayúsculas (`SI` y `NO`).
2. **`desaggregate_multiple_responses()`**: Desagrega variables de selección múltiple separadas por punto y coma (`;`) en indicadores binarios independientes (`0` o `1` tipo `Int64`, conservando `NaN` en registros nulos).
3. **`encode_dietary_frequencies()`**: Transforma las escalas ordinales cualitativas de frecuencia de consumo alimentario (`Nunca`=0, `Rara vez`=1, `A veces`=2, `Frecuentemente`=3, `Diario`=4).


In [5]:
# 1. Estandarizar columnas booleanas
df_proc = hs.standardize_boolean_columns(df_raw)

# 2. Desagregar respuestas múltiples (genera variables binarias independientes)
df_proc = hs.desaggregate_multiple_responses(
    df_proc,
    columns=["Salud_Transporte", "Salud_Agua", "Exposicion_Talleres", "Exposicion_Lugares", "Exposicion_Industrias"]
)

# 3. Codificar frecuencias de consumo dietario en escalas ordinales enteras
df_proc = hs.encode_dietary_frequencies(df_proc)

print(f"Dataset procesado con éxito: {df_proc.shape[0]} pacientes y {df_proc.shape[1]} variables estandarizadas.")

Dataset procesado con éxito: 48 pacientes y 56 variables estandarizadas.


--- 
## 5. Extracción de la Muestra Analítica y Filtrado de Metales

Separamos la **Población Total** ($N = 48$) de la **Muestra Analítica** ($n = 20$, aquellos niños con analíticas de laboratorio completas en sangre) mediante `get_analytical_sample()`.

Asimismo, si se requiere aislar exclusivamente un biomarcador de interés (ej. Plomo) eliminando los restantes para análisis focalizados, utilizamos `select_metal()`.


In [6]:
# 1. Extraer muestra analítica con mediciones de metales (n=20)
df_analytical = hs.get_analytical_sample(df_proc)
print(f"Muestra analítica extraída: {df_analytical.shape[0]} pacientes y {df_analytical.shape[1]} variables.")

# 2. Demostración de filtrado unimetal para Plomo (Pb)
df_pb = hs.select_metal(df_proc, concentration_col="plomo")
print(f"Dataset focalizado en Plomo: {df_pb.shape[0]} pacientes y {df_pb.shape[1]} variables.")

Muestra analítica extraída: 20 pacientes y 56 variables.
Dataset focalizado en Plomo: 20 pacientes y 52 variables.


--- 
## 6. Diagnóstico de Sesgo de Selección y Balance de Muestras

Para evaluar si el subconjunto de pacientes con mediciones de laboratorio ($n=20$) es representativo de la cohorte total no seleccionada ($n=28$), ejecutamos `compare_groups()`.

### Métricas de Evaluación
- **Diferencia Media Estandarizada (SMD)**:
  - $|\text{SMD}| \le 0.10$: Balance óptimo (sin sesgo relevante).
  - $0.10 < |\text{SMD}| \le 0.25$: Desequilibrio leve.
  - $|\text{SMD}| > 0.25$: **Desequilibrio sustancial / Sesgo de selección**.
- **Pruebas Estadísticas de Hipótesis**:
  - Variables continuas: Prueba $t$ de Welch para varianzas heterogéneas.
  - Variables categóricas: Prueba Exacta de Fisher (cuando frecuencias esperadas $E_{ij} < 5$) o Chi-cuadrado ($\chi^2$).
- **Implicación Metodológica**: Las variables desequilibradas identificadas (especialmente `Riesgo_Hg`, `Sector`, `Sexo`, `Es_Expuesto`) deben incluirse como **factores de ajuste obligatorio** en análisis bivariantes y regresiones multivariantes posteriores.


In [7]:
# Reporte integral de comparación y balance entre seleccionados vs no seleccionados
report_balance = hs.compare_groups(df_proc)
report_balance

<ComparationReport selected=20 non_selected=28>

--- 
## 7. Exportación de Resultados y Datasets

Todos los datasets procesados y reportes generados admiten exportación directa a múltiples formatos estructurados (CSV, Excel `.xlsx`, HTML interactivo Booktabs):


In [8]:
# 1. Guardar datasets limpios para auditoría y análisis posteriores
df_proc.to_csv("datos_poblacion_procesados_N48.csv", index=False, encoding="utf-8")
df_analytical.to_csv("datos_muestra_analitica_n20.csv", index=False, encoding="utf-8")

# 2. Exportar reportes de validación y balance a HTML interactivo y Excel
report_qc.to_html("reporte_calidad_datos.html", full_page=True)
report_balance.to_excel("reporte_balance_sesgo.xlsx")

print("¡Datasets procesados y reportes exportados exitosamente a CSV, Excel y HTML!")

¡Datasets procesados y reportes exportados exitosamente a CSV, Excel y HTML!
